# Sentimendianalüüs


## Sisukord

* [Ülesanne 9.1. ja Boonusülesanne](#9)
* [Sentimendianalüüs](#sentiment)
* [Andmete lugemine](#lugemine)
* [Sõnade relevantsuse hindamine](#relevants)
* [Logistiline regressioon dokumentide klassifitseerimiseks](#klf)
* [Ülesanne 9.2.](#9_2)

Põhineb S.Raschka *Python Machine Learnig* raamatu 
[peatükil 8](https://github.com/rasbt/python-machine-learning-book/blob/master/code/ch08) ([*MIT litsents*](https://github.com/rasbt/python-machine-learning-book/blob/master/LICENSE.txt)).

<a id='9'></a>


## Ülesanne 9.BOONUS

Koostada oma originaalandmetest (vt ka Moodle sektsiooni *Iseseisev töö*) .csv fail. Kirjeldada andmestikku lühidalt ja dokumenteerida andmestiku atribuudid eraldi failis kujul **atribuudi_nimi, andmetüüp (tõeväärtus, kategooria, täisarv, reaalarv, tekst, kuupäev,...), selgitus**. Kui on olemas klass või arvväärtus, mida soovime ennustada, siis dokumenteerida ka see.

Viige oma originaalandmestik sellisele kujule, mida ka teised tudengid saaksid kasutada ja laadige see Moodlesse üles. Vaadake, et teil oleksid seaduslikud  ja tööalased õigused neid andmeid jagada. Anonümiseerige isikuandmed (kui on) st asendage nimed ja isikukoodid meelevaldsete koodidega.

Kuigi iseseisvas töös on lubatud kasutada avalikke andmeid ka [Kagglest](https://www.kaggle.com/datasets) ja muudest allikatest, siis selle ülesandena lähevad kirja ainult andmed, mis pole avalikult saadaval või mis on saadud avalikest andmetest töötlemise abil. Näiteks Kagglest võetud .csv faili muutmata kujul esitamine ei ole selle boonusülesande mõte.


## Ülesanne 9.1.


Laadida Project Gutenberg lehelt alla Goethe Fausti http://www.gutenberg.org/ebooks/14591 ja Iliase http://www.gutenberg.org/ebooks/6130 tekstifailid (*plain text, utf-8*). Jagada need failid salmideks, kus salme eraldavad kaks reavahetust ja salmipikkus on üle saja ja alla tuhande tähemärgi, näiteks:

`text = file.read()`

`salmid = [t for t in text.split("\n\n") if 100 < len(t) < 1000]`

Koostage .csv fail, kus read on salmid ja esimeseks veeruks on salmi tekst ning teiseks veeruks on salmi allikas (Ilias või Faust).

In [22]:
import pandas as pd

with open('./Ilias.txt', 'r') as file:
    text_ilias = file.read()
    salmid_ilias = [t for t in text_ilias.split("\n\n") if 100 < len(t) < 1000][2:]
# salmid_ilias

with open('./faust.txt', 'r') as file:
    text_faust = file.read()
    salmid_faust = [t for t in text_faust.split("\n\n") if 100 < len(t) < 1000][3:]

allikas_faust = []
for i in range(len(salmid_faust)):
    allikas_faust += ['Faust']
    
allikas_ilias = []
for i in range(len(salmid_ilias)):
    allikas_ilias += ['Ilias']
    
salmid_df = pd.DataFrame({'Salm': [*salmid_faust, *salmid_ilias], 'Allikas': [*allikas_faust, *allikas_ilias]})

# salmid_df.to_csv('./salmid_csv.csv')
salmid_df



,Salm,Allikas
0,It is twenty years since I first determined to...,Faust
1,"[B] ""You are right,"" said Goethe; ""there are g...",Faust
2,"""The rhythm,"" said Goethe, ""is an unconscious ...",Faust
3,"""_All that is poetic in character should be ry...",Faust
4,"Tycho Mommsen, in his excellent essay, _Die Ku...",Faust
...,...,...
1988,International donations are gratefully accepte...,Ilias
1989,Please check the Project Gutenberg web pages f...,Ilias
1990,Professor Michael S. Hart was the originator o...,Ilias
1991,Project Gutenberg™ eBooks are often created fr...,Ilias


<a id='sentiment'></a>
## Sentimendianalüüs

Sentimendianalüüs on mingi teksti klassifitseerimine emotsionaalse sisu (positiivne-negatiivne, allikas1-allikas2 vms) alusel. Esmapilgul on probleemiks see, et tekst ei ole andmetabel, mis omaks arvväärtustega atribuute. Sentimendianalüüsiks teisendame teksti tavaliseks andmetabeliks, mille atribuudid vastavad sõnade sagedustele (terminisagedused). Seejärel on meil tegemist tavalise klassifitseerimisprobleemiga. 

Sentimendianalüüsi saab lihtaslt kasutada iseseisva töö jaoks originaalandmestiku genereerimiseks: võtta tekstid mitmest allikast, teisendada need terminisageduste abil andmetabeliks ja seejärel on meil lahendamiseks tüüpiline klassifitseerimisprobleem.

<a id='lugemine'></a>
## Andmete lugemine

Andmete töötluseks ja movie_data.csv faili koostamiseks on eraldi ipynb märkmik.

In [2]:
import numpy as np
import pandas as pd
import math
from sklearn.feature_extraction.text import CountVectorizer


In [3]:
# Eeldus: movie_data.csv on tekitatud

df = pd.read_csv('./movie_data.csv')
df.head(10)
# for text in df['review'].head(): print(text, '\n\n') # print all reviews

,review,sentiment
0,The House of the Spirits is a gripping tale of...,1
1,"I have very fond memories of this film, as I s...",1
2,Seeing this movie always reminds me of what I ...,1
3,"Originally filmed in 1999 as a TV pilot, ""Mulh...",1
4,but I want to say I cannot agree more with Moi...,1
5,"I don't know the stars, or modern Chinese teen...",1
6,Dominick (Nicky) Luciano wears a 'Hulk' T-shir...,1
7,Just a comment on New Orleans accents...<br />...,1
8,This is only the fourth effort I’ve watched fr...,0
9,"This time we get a psycho toy maker named ""Joe...",0


<a id='relevants'></a>
## Sõnade relevantsuse hindamine

Terminisagedus $tf(t, d)$ näitab kui mitu korda termin $t$ esineb dokumendis $d$. Klassi [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) transformeerimismeetod `fit_transform(X)` teisendab tekstimassiivi $t \times d$ terminisagedusmassiiviks.

In [4]:
count = CountVectorizer()
docs = np.array([
    'The sun is shining',
    'The weather is sweet',
    'The sun is shining and weather is sweet'])
bag = count.fit_transform(docs)

print(count.vocabulary_) # sõnastik näitab, mis indeksiga allolevas maatriksis vastav sõna on
print(bag.toarray()) # maatriks näitab mitu korda teatud sõna antud lauses on

{'the': 5, 'sun': 3, 'is': 1, 'shining': 2, 'weather': 6, 'sweet': 4, 'and': 0}
[[0 1 1 1 0 1 0]
 [0 1 0 0 1 1 1]
 [1 2 1 1 1 1 1]]


Sõnad, mis esinevad peaaegu kõigis dokumentides ei ole tavaliselt   dokumentide eristamise mõttes kuigi informatiivsed. Mõõt **idf** (*inverse document frequency*)  on logaritm dokumentide arvu $n_d$ suhtest sõna $t$ sisaldavate dokumentide arvu $f_d(t)$ ja on **seega seda suurem, mida vähemates dokumentides sõna esineb**. Vahel modiitseeritakse seda funktsiooni liites jagajale või jagatavale arvu 1.

$$ idf (t) = log \frac{n_d}{f_d(t)} $$

Mõõt **tf-idf** (*term frequency - inverse document frequency*) korrigeerib terminisagedust $tf$ mõõduga $idf$

$$ tfidf (t, d) = tf(i, d) \times idf(t) $$

In [5]:
np.set_printoptions(precision=2)

In [6]:
X = bag.toarray()

def idf(X):
    """ 
    Leiame idf  mõõtude vektori kõigile sõnadele kui meie 
    sagedusmaatriks (bag.toarray()) on X.
    """
    n_d = len(X)
    f_d = np.sum(X != 0, axis=0) # veergude kaupa summeerime, mitmes dokumendis ehk antud näites mitmes reas ei võrdu antud sõna arv 0-ga
    #print(f_d)
    return np.log(n_d / f_d)

idf(X) # leiame idf-i iga sõna jaoks (veeruna)


array([1.1 , 0.  , 0.41, 0.41, 0.41, 0.  , 0.41])

In [7]:
def tfidf(X):
    """ 
    Leiame tfidf  sageduste vektori kõigile sõnadele kui meie 
    sagedusmaatriks (bag.toarray()) on X.
    """
    return X * idf(X)

tfidf(X) # korrutame iga sõna sageduse dokumendis talle vastava idf-ga läbi

array([[0.  , 0.  , 0.41, 0.41, 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.41, 0.  , 0.41],
       [1.1 , 0.  , 0.41, 0.41, 0.41, 0.  , 0.41]])

Moodul sklearn pakub selleks klassi [TfidfTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html). Tulemused erinevad seoses funktsioonide veidi  erinevate definitsioonidega 

$$ idf (t) = log \frac{1+n_d}{1+f_d(t)} $$


$$ tfidf (t, d) = tf(i, d) \times (idf(t) + 1) $$

ja dokumendi sagedusvektori normaliseerimisega ühikpikkuseks (L2-normaliseerimine).

In [8]:
from sklearn.feature_extraction.text import TfidfTransformer
tfidf_t = TfidfTransformer()
TFIDF_X = tfidf_t.fit_transform(count.fit_transform(docs)).toarray()
TFIDF_X

array([[0.  , 0.43, 0.56, 0.56, 0.  , 0.43, 0.  ],
       [0.  , 0.43, 0.  , 0.  , 0.56, 0.43, 0.56],
       [0.44, 0.53, 0.34, 0.34, 0.34, 0.26, 0.34]])

In [9]:
np.sum(TFIDF_X**2, axis=1)

array([1., 1., 1.])

<a id='klf'></a>
## Logistiline regressioon dokumentide klassifitseerimiseks





Klass [TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) lihtsustab tf-idf mõõdu leidmist olles samaväärne  [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) ja [TfidfTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html) üksteise järel rakendamisega. Alustuseks leiame hüperparameetrite otsinguga [GrisSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.htm) parimad parameetrid. Kasutame selleks ainult 5000 objekti, muidu oleks see samm väga ajamahukas.

In [10]:
len(df)

50000

In [11]:
X_train = df.loc[:5000, 'review'].values
y_train = df.loc[:5000, 'sentiment'].values
X_test = df.loc[5000:10000, 'review'].values
y_test = df.loc[5000:10000, 'sentiment'].values

In [12]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
import string

In [13]:
print(ord(".")) # ord() annab märgi ordinaalväärtused vms, 44 on koma ja 46 on punkt
"k,k,l,o,ooo.".translate({44: None, 46: "@"}) # eemaldame komad ja asendame punktid @ märgiga, kasutades nende oridnaalväärtuseid

46


'kkloooo@'

In [14]:
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [15]:
#{ord(c):None for c in string.punctuation} # eemaldab kõik kirjavahemärgid

In [16]:
tfidf = TfidfVectorizer(strip_accents=None,
                        lowercase=False,
                        preprocessor=None)

def tokenizer(text):return text.split()
def nopunctuation_tokenizer(text):
    np_text = text.translate({ord(c):None for c in string.punctuation}) # kui ei eemalda kirjavahemärke, siis iga sõna, mille järel on märk, läheb eraldi sõnana kirja - pole korrektne 
    return np_text.split()

param_grid = [{'vect__ngram_range': [(1, 1)],
               'vect__tokenizer': [tokenizer, nopunctuation_tokenizer],
               'vect__token_pattern': [None],
               'clf__penalty': ['l1', 'l2'],
               'clf__C': [1.0, 10.0, 100.0]},
              {'vect__ngram_range': [(1, 1)],
               'vect__tokenizer': [tokenizer, nopunctuation_tokenizer],
               'vect__token_pattern': [None],
               'vect__use_idf':[False],
               'vect__norm':[None],
               'clf__penalty': ['l1', 'l2'],
               'clf__C': [1.0, 10.0, 100.0]},
              ]

lr_tfidf = Pipeline([('vect', tfidf),
                     ('clf', LogisticRegression(random_state=0, solver="liblinear"))])

gs_lr_tfidf = GridSearchCV(lr_tfidf, param_grid,
                           scoring='accuracy',
                           cv=5,
                           verbose=1,
                           n_jobs=1)

In [33]:
# Väga ajamahukas samm
gs_lr_tfidf.fit(X_train, y_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x000001AF839FB260>>
Traceback (most recent call last):
  File "c:\Users\Maive\miniconda3\envs\andmekaeve_suurandmetest\Lib\site-packages\ipykernel\ipkernel.py", line 785, in _clean_thread_parent_frames
    active_threads = {thread.ident for thread in threading.enumerate()}
                                                 ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Maive\miniconda3\envs\andmekaeve_suurandmetest\Lib\threading.py", line 1534, in enumerate
    def enumerate():
    
KeyboardInterrupt: 
c:\Users\Maive\miniconda3\envs\andmekaeve_suurandmetest\Lib\site-packages\sklearn\svm\_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
print("Parimad parameetrid: ", gs_lr_tfidf.best_params_)
print("\nTäpsus: ", gs_lr_tfidf.best_score_)


Parimad parameetrid:  {'clf__C': 100.0, 'clf__penalty': 'l2', 'vect__ngram_range': (1, 1), 'vect__token_pattern': None, 'vect__tokenizer': <function nopunctuation_tokenizer at 0x00000203586E93A0>}

Täpsus:  0.8582253746253746


Seejärel kasutame neid parameetreid ennustava mudeli treenimiseks andmestiku esimese 25 000 objekti peal ja leiame kõige suurema koefitsendiga (kõige kaalukamad) sõnad otsuse tegemisel.

In [ ]:
X_train = df.loc[:25000, 'review'].values
y_train = df.loc[:25000, 'sentiment'].values
X_test = df.loc[25000:, 'review'].values
y_test = df.loc[25000:, 'sentiment'].values

lr_tfidf.set_params(**gs_lr_tfidf.best_params_)
lr_tfidf.fit(X_train, y_train)

Pipeline(steps=[('vect',
                 TfidfVectorizer(lowercase=False, token_pattern=None,
                                 tokenizer=<function nopunctuation_tokenizer at 0x00000203586E93A0>)),
                ('clf',
                 LogisticRegression(C=100.0, random_state=0,
                                    solver='liblinear'))])

Teeme 10-kordse ristkontrolli.

In [ ]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(estimator=lr_tfidf,
                         X=X_train,
                         y=y_train,
                         cv=10,
                         n_jobs=1)
print('CV täpsused: ', scores)
print('CV keskmine täpsus: %.3f' % np.mean(scores), "+/- %.3f" % np.std(scores))

CV täpsused:  [0.89 0.89 0.88 0.88 0.9  0.88 0.9  0.89 0.9  0.9 ]
CV keskmine täpsus: 0.892 +/- 0.007


Ja lõpuks kontrollime erinevate meetrikate ja testandmetega.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import confusion_matrix

y_pred = lr_tfidf.predict(X_test)
print("Täpsus testandmetel:", accuracy_score(y_test, y_pred))
print("F1-skoor testandmetel:", f1_score(y_test, y_pred))
print("Eksimismaatriks:")
print(confusion_matrix(y_test, y_pred))

Täpsus testandmetel: 0.89476
F1-skoor testandmetel: 0.8951082406410716
Eksimismaatriks:
[[11143  1405]
 [ 1226 11226]]


In [ ]:
coef = lr_tfidf.named_steps['clf'].coef_[0] # annab logitilise regressioonil iga sõna koefitsiendid ehk sõna olulisus klassi ennustamisel
print("Koefitsendid:", coef)

Koefitsendid: [-0.04 -4.86  1.09 ...  0.06 -0.01 -2.4 ]


In [ ]:
word_dict = lr_tfidf.named_steps['vect'].vocabulary_
print("Sõnastik:", len(word_dict)) # annab mitu sõna on sõnastikus ehk meie algandmetes

Sõnastik: 145431


In [ ]:
term_coef = [(coef[word_dict[w]], w) for w in word_dict]
term_coef = sorted(term_coef, key=lambda pair: abs(pair[0]), reverse=True)
print("Need sõnad mõjutavad arvustuse sentimenti enim (koefitsent, sõna):\n")
term_coef[:20]


Need sõnad mõjutavad arvustuse sentimenti enim (koefitsent, sõna):



[(-30.163405140363256, 'worst'),
 (-23.484700985222013, 'awful'),
 (-21.420029416165136, 'waste'),
 (21.331065960953826, '710'),
 (18.210690229167227, 'excellent'),
 (-16.629842670432787, 'bad'),
 (-16.576548522894914, 'fails'),
 (-16.42485928512674, 'boring'),
 (16.272315235901335, 'great'),
 (-15.785860680556352, 'nothing'),
 (15.763725870990394, 'wonderful'),
 (-15.704029862739269, 'disappointing'),
 (-14.80804035948606, 'poorly'),
 (-14.598099772128512, 'poor'),
 (-14.326725256635525, 'worse'),
 (-14.226544332275056, 'terrible'),
 (-14.197902733280756, 'dull'),
 (13.881969626133381, '810'),
 (-13.855992129665413, 'disappointment'),
 (-13.708671589825316, 'forgettable')]

Üldiselt vastavad kaalud sõnade emotsionaalsele sisule. 

<a id='9_2'></a>

## Ülesanne 9.2.

Looge konveier, mis koosneb `TfidfVectorizer` ja `LogisticRegression` sammudest. Jagage ülesandes 9.1 loodud andmestik treening- ja testandmeteks. Klassid võiksid olla kahendkujul, näiteks `df['allikas']=='faust'`. Andmeteks on tekstiatribuut (näiteks `df['tekst']`) ja see on sisend `TfidfVectorizer` eeltöötlusele,  seega seda varem töödelda pole vaja. Treenige konveier treeningandmetel ja leidke testandmetel selle täpsus ja F1 skoor. 

In [23]:
# Ennustada teksti allikat
allika_map = {"Faust": 1, "Ilias": 0}
salmid_df["Allikas"] = salmid_df["Allikas"].map(allika_map)
salmid_df

,Salm,Allikas
0,It is twenty years since I first determined to...,1
1,"[B] ""You are right,"" said Goethe; ""there are g...",1
2,"""The rhythm,"" said Goethe, ""is an unconscious ...",1
3,"""_All that is poetic in character should be ry...",1
4,"Tycho Mommsen, in his excellent essay, _Die Ku...",1
...,...,...
1988,International donations are gratefully accepte...,0
1989,Please check the Project Gutenberg web pages f...,0
1990,Professor Michael S. Hart was the originator o...,0
1991,Project Gutenberg™ eBooks are often created fr...,0


In [26]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Jagan andmestiku test- ja treeningandmeteks
X_train, X_test, y_train, y_test = train_test_split(salmid_df['Salm'], 
                                                    salmid_df['Allikas'], 
                                                    test_size=0.2, 
                                                    random_state=1)

In [28]:
# Loon konveiere, mis koosneb `TfidfVectorizer` ja `LogisticRegression` sammuudes
def tokenizer(text):return text.split()
def nopunctuation_tokenizer(text):
    np_text = text.translate({ord(c):None for c in string.punctuation})
    return np_text.split()

param_grid = [{'vect__ngram_range': [(1, 1)],
               'vect__tokenizer': [tokenizer, nopunctuation_tokenizer],
               'vect__token_pattern': [None],
               'clf__penalty': ['l1', 'l2'],
               'clf__C': [1.0, 10.0, 100.0]},
              {'vect__ngram_range': [(1, 1)],
               'vect__tokenizer': [tokenizer, nopunctuation_tokenizer],
               'vect__token_pattern': [None],
               'vect__use_idf':[False],
               'vect__norm':[None],
               'clf__penalty': ['l1', 'l2'],
               'clf__C': [1.0, 10.0, 100.0]},
              ]

pipe_Tfidf_lr = Pipeline([ ('vect', TfidfVectorizer(strip_accents=None,
                        lowercase=False,
                        preprocessor=None)), 
                         ('clf', LogisticRegression(random_state=0, solver="liblinear"))])


gs_pipe_Tfidf_lr = GridSearchCV(pipe_Tfidf_lr, param_grid,
                           scoring='accuracy',
                           cv=5,
                           verbose=1,
                           n_jobs=1)

In [29]:
#Viin GridSearchi läbi

gs_pipe_Tfidf_lr.fit(X_train, y_train)
print("Parimad parameetrid: ", gs_pipe_Tfidf_lr.best_params_)
print("\nTäpsus: ", gs_pipe_Tfidf_lr.best_score_)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


c:\Users\Maive\miniconda3\envs\andmekaeve_suurandmetest\Lib\site-packages\sklearn\svm\_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


Parimad parameetrid:  {'clf__C': 100.0, 'clf__penalty': 'l2', 'vect__ngram_range': (1, 1), 'vect__token_pattern': None, 'vect__tokenizer': <function nopunctuation_tokenizer at 0x000001AF8738E7A0>}

Täpsus:  0.9410303424617024


In [30]:
pipe_Tfidf_lr.set_params(**{'clf__C': 100.0, 'clf__penalty': 'l2', 'vect__ngram_range': (1, 1), 'vect__token_pattern': None, 'vect__tokenizer': nopunctuation_tokenizer})
pipe_Tfidf_lr.fit(X_train, y_train)

Pipeline(steps=[('vect',
                 TfidfVectorizer(lowercase=False, token_pattern=None,
                                 tokenizer=<function nopunctuation_tokenizer at 0x000001AF8738E7A0>)),
                ('clf',
                 LogisticRegression(C=100.0, random_state=0,
                                    solver='liblinear'))])

In [35]:
# Treenige konveier treeningandmetel ja leidke testandmetel 
# selle täpsus ja F1 skoor. 

# Konveieri täpsus ja F1 skoor

from sklearn.metrics import accuracy_score, f1_score
from sklearn.metrics import confusion_matrix

y_pred = pipe_Tfidf_lr.predict(X_test)
print("Täpsus testandmetel:", accuracy_score(y_test, y_pred))
print("F1-skoor testandmetel:", f1_score(y_test, y_pred))
print("Eksimismaatriks:")
print(confusion_matrix(y_test, y_pred))


Täpsus testandmetel: 0.9423558897243107
F1-skoor testandmetel: 0.8977777777777778
Eksimismaatriks:
[[275  10]
 [ 13 101]]


In [36]:
# # Vaatasin lisaks: 
# coef = pipe_Tfidf_lr.named_steps['clf'].coef_[0] # annab logitilise regressioonil iga sõna koefitsiendid ehk sõna olulisus klassi ennustamisel
# print("Koefitsendid:", coef)

# word_dict = pipe_Tfidf_lr.named_steps['vect'].vocabulary_
# print("Sõnastik:", len(word_dict)) # annab mitu sõna on sõnastikus ehk meie algandmetes

# term_coef = [(coef[word_dict[w]], w) for w in word_dict]
# term_coef = sorted(term_coef, key=lambda pair: abs(pair[0]), reverse=True)
# print("Need sõnad mõjutavad arvustuse sentimenti enim (koefitsent, sõna):\n")
# term_coef[:20]

Koefitsendid: [-0.08 -0.35 -0.05 ... -0.09 -0.69 -0.3 ]
Sõnastik: 14461
Need sõnad mõjutavad arvustuse sentimenti enim (koefitsent, sõna):



[(-10.722271892262192, 'the'),
 (-9.819883921421065, 'of'),
 (8.840443687420155, 'I'),
 (7.735389832172017, 'me'),
 (-7.191409005867449, 'his'),
 (-7.099205141171763, 'p'),
 (-6.781394803133989, 'thy'),
 (-6.4382322614892376, 'Homer'),
 (-5.69960296413676, 'to'),
 (-5.681799801366704, 'gods'),
 (-5.479094378117168, 'their'),
 (5.255613233558308, 'us'),
 (-5.144504373708159, 'by'),
 (5.102361818215944, 'Tis'),
 (-5.060903554900629, 'e'),
 (-4.971626429519331, 'thus'),
 (-4.819586078964252, 'Hector'),
 (4.812122497994258, 'As'),
 (-4.811034029068916, 'arms'),
 (-4.67789328002596, 'our')]